In [1]:
#Membuat dataset
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [52]:
# Persiapan spark untuk tugas
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count

spark = SparkSession.builder.appName("Tugas6").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("Siap untuk mengerjakan tugas.")

Siap untuk mengerjakan tugas.


In [3]:
##A. EXTRACT
#Baca ketiga sumber data jadi tiga dataframe terpisah. Tampilkan jumlah baris dan printSchema()
#Membaca file csv dan json
df_transaksi=spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_ulasan=spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)
df_produk=spark.read.json("tugas6_produk.json")

#Menampilkan Schema dan jumlah baris dataframe
print("data transaksi = ",df_transaksi.count())
df_transaksi.printSchema()
print("data ulasan = ",df_ulasan.count())
df_ulasan.printSchema()
print("data produk = ",df_produk.count())
df_produk.printSchema()

data transaksi =  5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

data ulasan =  3500
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)

data produk =  30
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)



In [24]:
##B. TRANSFORM (part 1)
#Gabungkan ketiga dataframe
#transaksi ke ulasan (tidak semua transaksi punya ulasan)
#transaksi ke produk (setiap transaksi pasti punya produk valid)
df_gabung= df_transaksi.join(
    df_ulasan, on="order_id", how="left").join(
    df_produk, on="product_id", how="inner")
df_gabung = df_gabung.withColumn("total_pendapatan", col("unit_terjual")*col("harga"))
df_gabung.show(10)
print("jumlah baris jika digabung memakai left: ", df_gabung.count())

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
|        10|     TX5|           3|2026-10-01 00:00:00|     5| 75000|  Elektronik|  Produk-10|          225000|
|

In [23]:
df_gabung2= df_transaksi.join(
    df_ulasan, on="order_id", how="inner").join(
    df_produk, on="product_id", how="inner")
df_gabung2 = df_gabung.withColumn("total_pendapatan", col("unit_terjual")*col("harga"))
df_gabung2.show(10)
print("jumlah baris jika digabung memakai inner: ", df_gabung.count())

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        10|     TX5|           3|2026-10-01 00:00:00|     5| 75000|  Elektronik|  Produk-10|          225000|
|        26|     TX6|           6|2026-10-24 00:00:00|     4| 75000|   Kesehatan|  Produk-26|          450000|
|         7|     TX8|           2|2026-10-09 00:00:00|     5| 75000|  Elektronik|   Produk-7|          150000|
|

In [25]:
##C. TRANSFORM (part 2)
#Menambahkan kolom ada ulasan bernilai True/False
from pyspark.sql.functions import when
df_gabung = df_gabung.withColumn(
    "ada_ulasan",
    when(col("rating") != 0, "True")
    .otherwise("False")
)
#Mengisi kolom yang null dengan rating 0

df_gabung = df_gabung.na.fill(value=0, subset= "rating").show(10)


+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|ada_ulasan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|      True|
|         8|     TX1|           6|2026-10-25 00:00:00|     0|250000|     Fashion|   Produk-8|         1500000|     False|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|      True|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|      True|
|        19|     TX4|           6|2026-10-07 00:00:00|     0| 75000|Rumah Tangga|  Produk-19|          450000|     False|
|        10|     TX5|   

In [36]:
##D. LOAD
#Menyimpan hasil akhir ke hdfs dalam forma Parquet, dipartisi kategori ke path user/mahasiswa/tugas6/hasil_etl
!hdfs dfs -mkdir -p /user/mahasiswa/tugas6
print("direktori berhasil dibuat di HDFS")

direktori berhasil dibuat di HDFS


In [82]:
df_gabung.write.mode("overwrite").partitionBy("kategori").parquet(
    "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"
)

print("Pipeline ETL selesai, hasil tersimpan di HDFS")
!hdfs dfs -ls  -R /user/mahasiswa/tugas6/hasil_etl

Pipeline ETL selesai, hasil tersimpan di HDFS


-rw-r--r--   3 celloz supergroup          0 2026-09-24 11:34 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - celloz supergroup          0 2026-09-24 11:34 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 celloz supergroup      16448 2026-09-24 11:34 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik/part-00000-f35080b3-8bb1-422a-8099-5637b397e12e.c000.snappy.parquet
drwxr-xr-x   - celloz supergroup          0 2026-09-24 11:34 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 celloz supergroup      11208 2026-09-24 11:34 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion/part-00000-f35080b3-8bb1-422a-8099-5637b397e12e.c000.snappy.parquet
drwxr-xr-x   - celloz supergroup          0 2026-09-24 11:34 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 celloz supergroup      10464 2026-09-24 11:34 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan/part-00000-f35080b3-8bb1-422a-8099-5637b397e12e.c000.snappy.parquet
drwxr-xr-x   - ce

In [54]:
#Membaca kembali file dari HDFS untuk verifikasi
df_verif = spark.read.parquet("hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl")
print("Total jumlah total record = ",df_verif.count())
df_verif.show(10)

Total jumlah total record =  5000
+----------+--------+------------+-------------------+------+------+-----------+----------------+----------+----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|nama_produk|total_pendapatan|ada_ulasan|  kategori|
+----------+--------+------------+-------------------+------+------+-----------+----------------+----------+----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Produk-21|           50000|      True|Elektronik|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Produk-25|          100000|      True|Elektronik|
|        10|     TX5|           3|2026-10-01 00:00:00|     5| 75000|  Produk-10|          225000|      True|Elektronik|
|         7|     TX8|           2|2026-10-09 00:00:00|     5| 75000|   Produk-7|          150000|      True|Elektronik|
|        30|     TX9|           1|2026-10-27 00:00:00|     0|250000|  Produk-30|          250000|     False|Elektronik|
|     

In [55]:
## E. Insight Akhir
# Menampilkan kategori produk yang memiliki persentase transaksi 
# dengan ulasan (ada_ulasan =True) paling rendah

df_insight =df_verif.groupBy("kategori").agg(
    spark_sum(when(col("ada_ulasan")=="True",1)
        .otherwise(0)).alias("jumlah_ulasan_true"))

#Menambahkan kolom persentase                                            
df_insight = df_insight.withColumn("persentase_ulasan_transaksi", 
            col("jumlah_ulasan_true") / 5000 * 100)
df_insight.show()

+------------+------------------+---------------------------+
|    kategori|jumlah_ulasan_true|persentase_ulasan_transaksi|
+------------+------------------+---------------------------+
|  Elektronik|              1168|                      23.36|
|     Fashion|               726|                      14.52|
|   Kesehatan|               662|         13.239999999999998|
|Rumah Tangga|               575|                       11.5|
|     Makanan|               369|          7.380000000000001|
+------------+------------------+---------------------------+



In [51]:
spark.stop()
print("spark berhenti disini")

spark berhenti disini
